# ETL para Power BI

Genera tablas limpias: accidentes, vehículos y tipos de accidente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Proyecto final Keepcoding/datasets/processed data/accidentes_madrid_procesado.csv")
df.head()

,num_expediente,gravedad,fecha,hora,distrito,estado_meteorológico,temperature_2m,precipitation,windspeed_10m,coordenada_x_utm,...,tipo_persona_Pasajero,tipo_persona_Peatón,tipo_persona_Peatón (atropello sc),fecha_date,festivo,estacion,fin_semana,hora_punta,noche,franja_horaria
0,2020S000006,0,2020-01-01,5,CARABANCHEL,Despejado,0.2,0.0,6.6,436672.459,...,1,0,0,2020-01-01,1,invierno,0,0,1,madrugada
1,2020S000007,1,2020-01-01,6,SALAMANCA,Despejado,-0.6,0.0,6.1,443740.487,...,1,0,0,2020-01-01,1,invierno,0,0,1,madrugada
2,2020S000015,0,2020-01-01,2,CHAMARTÍN,Despejado,1.3,0.0,7.2,441461.761,...,1,0,0,2020-01-01,1,invierno,0,0,1,madrugada
3,2020S000017,0,2020-01-01,3,PUENTE DE VALLECAS,Despejado,1.5,0.0,6.2,442141.822,...,0,0,0,2020-01-01,1,invierno,0,0,1,madrugada
4,2020S000020,1,2020-01-01,9,CARABANCHEL,Despejado,-1.6,0.0,7.4,439691.436,...,1,0,0,2020-01-01,1,invierno,0,1,0,mañana


In [ ]:
# --- TRANSFORMACIÓN COORDENADAS UTM A LAT/LON ---
!pip install pyproj

Transformación de coordenadas_x y y a latitud y longitud

In [ ]:


from pyproj import Transformer

# Crear transformador (España → WGS84)
transformer = Transformer.from_crs("epsg:25830", "epsg:4326", always_xy=True)

def utm_to_latlon(x, y):
    lon, lat = transformer.transform(x, y)
    return lat, lon

# Aplicar
df[["lat", "lon"]] = df.apply(
    lambda row: pd.Series(
        utm_to_latlon(row["coordenada_x_utm"], row["coordenada_y_utm"])
    ),
    axis=1
)

## Tabla accidentes

In [ ]:
df["id_accidente"] = df["num_expediente"]

cols_acc = [
    "id_accidente","gravedad","fecha","hora","distrito",
    "estado_meteorológico","temperature_2m","precipitation",
    "festivo","fin_semana","hora_punta",
    "lat","lon"
]

tabla_accidentes = df[cols_acc].drop_duplicates()
tabla_accidentes.head()

,id_accidente,gravedad,fecha,hora,distrito,estado_meteorológico,temperature_2m,precipitation,festivo,fin_semana,hora_punta,lat,lon
0,2020S000006,0,2020-01-01,5,CARABANCHEL,Despejado,0.2,0.0,1,0,0,40.380136,-3.746040
1,2020S000007,1,2020-01-01,6,SALAMANCA,Despejado,-0.6,0.0,1,0,0,40.428774,-3.663252
2,2020S000015,0,2020-01-01,2,CHAMARTÍN,Despejado,1.3,0.0,1,0,0,40.454145,-3.690376
3,2020S000017,0,2020-01-01,3,PUENTE DE VALLECAS,Despejado,1.5,0.0,1,0,0,40.374184,-3.681548
4,2020S000020,1,2020-01-01,9,CARABANCHEL,Despejado,-1.6,0.0,1,0,1,40.392031,-3.710600


## Tabla vehículos

In [ ]:
vehiculo_cols = [c for c in df.columns if c.startswith("vehiculo_")]

tabla_vehiculos = df[["id_accidente"] + vehiculo_cols].melt(
    id_vars="id_accidente",
    value_vars=vehiculo_cols,
    var_name="tipo_vehiculo",
    value_name="presente"
)

tabla_vehiculos = tabla_vehiculos[tabla_vehiculos["presente"] == 1]
tabla_vehiculos["tipo_vehiculo"] = tabla_vehiculos["tipo_vehiculo"].str.replace("vehiculo_", "")

tabla_vehiculos.head()

,id_accidente,tipo_vehiculo,presente
3,2020S000017,coche,1
9,2020S000035,coche,1
14,2020S000044,coche,1
15,2020S000047,coche,1
19,2020S000053,coche,1


## Agrupación

In [ ]:
def agrupar(x):
    if x in ["coche","furgoneta"]:
        return "Ligero"
    elif x in ["moto","bicicleta"]:
        return "Vulnerable"
    elif x in ["bus"]:
        return "Transporte público"
    elif x in ["camion"]:
        return "Pesado"
    else:
        return "Otros"

tabla_vehiculos["vehiculo_grupo"] = tabla_vehiculos["tipo_vehiculo"].apply(agrupar)
tabla_vehiculos.head()

,id_accidente,tipo_vehiculo,presente,vehiculo_grupo
3,2020S000017,coche,1,Ligero
9,2020S000035,coche,1,Ligero
14,2020S000044,coche,1,Ligero
15,2020S000047,coche,1,Ligero
19,2020S000053,coche,1,Ligero


## Tipo accidente

In [ ]:
acc_cols = [c for c in df.columns if c.startswith("tipo_accidente_")]

tabla_tipo_accidente = df[["id_accidente"] + acc_cols].melt(
    id_vars="id_accidente",
    value_vars=acc_cols,
    var_name="tipo_accidente",
    value_name="presente"
)

tabla_tipo_accidente = tabla_tipo_accidente[tabla_tipo_accidente["presente"] == 1]
tabla_tipo_accidente["tipo_accidente"] = tabla_tipo_accidente["tipo_accidente"].str.replace("tipo_accidente_", "")

tabla_tipo_accidente.head()

,id_accidente,tipo_accidente,presente
6,2020S000022,Alcance,1
7,2020S000025,Alcance,1
13,2020S000043,Alcance,1
18,2020S000050,Alcance,1
20,2020S000054,Alcance,1


# Tipo Persona

In [ ]:
# --- TABLA PERSONAS ---

# columnas de tipo de persona
persona_cols = [c for c in df.columns if c.startswith("tipo_persona_")]

tabla_personas = df[["id_accidente"] + persona_cols].melt(
    id_vars="id_accidente",
    value_vars=persona_cols,
    var_name="tipo_persona",
    value_name="presente"
)

# quedarse solo con las categorías activas
tabla_personas = tabla_personas[tabla_personas["presente"] == 1].copy()

# limpiar nombre de la categoría
tabla_personas["tipo_persona"] = tabla_personas["tipo_persona"].str.replace("tipo_persona_", "", regex=False)

# columnas de sexo en one-hot
sexo_cols = [c for c in df.columns if c.startswith("sexo_")]

tabla_sexo = df[["id_accidente"] + sexo_cols].melt(
    id_vars="id_accidente",
    value_vars=sexo_cols,
    var_name="sexo",
    value_name="presente"
)

tabla_sexo = tabla_sexo[tabla_sexo["presente"] == 1].copy()
tabla_sexo["sexo"] = tabla_sexo["sexo"].str.replace("sexo_", "", regex=False)

# deduplicar por seguridad
tabla_sexo = tabla_sexo[["id_accidente", "sexo"]].drop_duplicates()

# unir sexo
tabla_personas = tabla_personas.merge(
    tabla_sexo,
    on="id_accidente",
    how="left"
)

# añadir variables adicionales útiles para Power BI
extra_cols = [
    "id_accidente",
    "estacion",
    "alcohol_positivo",
    "edad_media",
    "gravedad"
]

tabla_personas = tabla_personas.merge(
    df[extra_cols].drop_duplicates(),
    on="id_accidente",
    how="left"
)

# ordenar columnas
tabla_personas = tabla_personas[
    [
        "id_accidente",
        "tipo_persona",
        "sexo",
        "estacion",
        "alcohol_positivo",
        "edad_media",
        "gravedad"
    ]
]

tabla_personas.head()

,id_accidente,tipo_persona,sexo,estacion,alcohol_positivo,edad_media,gravedad
0,2020S000006,Conductor,Hombre,invierno,1,32.5,0
1,2020S000006,Conductor,Mujer,invierno,1,32.5,0
2,2020S000007,Conductor,Hombre,invierno,0,32.0,1
3,2020S000007,Conductor,Mujer,invierno,0,32.0,1
4,2020S000015,Conductor,Hombre,invierno,0,39.5,0


## Exportar

In [ ]:
# Guardar archivos
tabla_accidentes.to_csv("/content/drive/MyDrive/Proyecto final Keepcoding/datasets/power bi data/tabla_accidentes.csv", index=False, encoding="utf-8-sig")
tabla_vehiculos.to_csv("/content/drive/MyDrive/Proyecto final Keepcoding/datasets/power bi data/tabla_vehiculos.csv", index=False, encoding="utf-8-sig")
tabla_tipo_accidente.to_csv("/content/drive/MyDrive/Proyecto final Keepcoding/datasets/power bi data/tabla_tipo_accidente.csv", index=False, encoding="utf-8-sig")
tabla_personas.to_csv("/content/drive/MyDrive/Proyecto final Keepcoding/datasets/power bi data/tabla_personas.csv", index=False, encoding="utf-8-sig")

print("Exportado correctamente en 'power bi data'")

Exportado correctamente en 'power bi data'
